<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/qwen_3_tts_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⚙️ BƯỚC 1: CÀI ĐẶT THƯ VIỆN & TẢI MODEL AI
import os
from IPython.display import Audio, display, clear_output

print("⏳ Đang cài đặt thư viện (Chỉ mất 1-2 phút)...")
os.system('pip install -U qwen-tts huggingface_hub pydub')
os.system('apt-get install -y ffmpeg sox libsox-fmt-all')
clear_output()
print("✅ Cài đặt xong thư viện!")

from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import gc
import re
from pydub import AudioSegment

torch.backends.cudnn.benchmark = True
current_model = None
current_model_type = None

def load_model(task_type):
    global current_model, current_model_type
    model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign" if task_type == "DESIGN" else "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

    if current_model_type == task_type and current_model is not None:
        return current_model

    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model {task_type}... (Vui lòng đợi)")
    current_model = Qwen3TTSModel.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda:0", attn_implementation="sdpa")
    current_model_type = task_type
    return current_model

print("✅ Hệ thống đã sẵn sàng!")

✅ Cài đặt xong thư viện!

********
********
 
✅ Hệ thống đã sẵn sàng!


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# @title 🎙️ BƯỚC 2: TẠO GIỌNG MẪU NAM & NỮ

# --- BẠN CÓ THỂ SỬA TEXT Ở ĐÂY HOẶC GIỮ NGUYÊN ---
nam_prompt = "A professional male podcast host, deep and soothing voice. Laughing and energetic."
nam_text = "Haha! Hello everyone, welcome to the show."

nu_prompt = "A warm female voice, soft and velvety. American accent. Very happy."
nu_text = "Wow! I am so excited to be here."
# --------------------------------------------------

print("⏳ Đang tạo giọng Nam...")
model = load_model("DESIGN")
with torch.inference_mode():
    w_nam, sr_nam = model.generate_voice_design(text=nam_text, instruct=nam_prompt)
sf.write("nam_ref.wav", w_nam[0], sr_nam)

print("⏳ Đang tạo giọng Nữ...")
with torch.inference_mode():
    w_nu, sr_nu = model.generate_voice_design(text=nu_text, instruct=nu_prompt)
sf.write("nu_ref.wav", w_nu[0], sr_nu)

clear_output()
print("✅ Đã tạo xong 2 giọng mẫu!")
print("👨 Giọng Nam (nam_ref.wav):")
display(Audio("nam_ref.wav"))
print("👩 Giọng Nữ (nu_ref.wav):")
display(Audio("nu_ref.wav"))

In [ ]:
# @title 🚀 BƯỚC 3: RENDER PODCAST SIÊU TỐC

# 👇👇👇 DÁN KỊCH BẢN CỦA BẠN VÀO ĐÂY (Giữa 3 dấu ngoặc kép) 👇👇👇
kich_ban = """
Nam: Hello everyone and welcome back to the Just English Channel, the best place on the internet to learn English naturally. I am your host, David.
Nu: And I am Emma. Welcome back, listeners. We are so incredibly happy to have you here with us again for another long episode. Today, we have a very fun, relaxing, and universally interesting topic to discuss. We are going to talk about hobbies.
Nam: Hobbies. Yes, absolutely the best part of life. The things we do when nobody is forcing us to work, pay bills, or study for exams. It is my absolute favorite topic because I consider myself an expert in having free time.
Nu: Well, that is exactly what a hobby is. A hobby is a regular activity done for enjoyment, typically during your leisure time. For our A2 and B1 learners, "leisure time" simply means your free time. It is the time when you are not working, not going to school, and not doing household chores.
Nam: If a hobby is an activity I do in my leisure time for pure enjoyment, then my number one, absolute favorite hobby is definitely sleeping. I practice it every single day. I am highly skilled at it. I can sleep on the sofa, I can sleep on a noisy bus, and sometimes I even practice this hobby at my desk.
Nu: Please do not sleep at your desk, David. That is called being a terrible employee, not having a hobby. Sleeping is a biological necessity. You need it to survive, just like breathing or drinking water. A hobby is something active that you choose to do to develop a new skill, create something, or relax your mind in a productive way.
Nam: I disagree completely. I actively choose to take naps. It takes dedication. And my mind is very relaxed when I am asleep. Therefore, taking naps is a lifestyle and a hobby. I am a professional napper.
Nu: You are impossible. A real hobby is something like reading, painting, gardening, or learning a new language. These activities improve you as a person. They help you grow, they keep your brain sharp, and they make you a more interesting person to talk to.
Nam: Okay, fine. If sleeping does not count as a valid hobby in your strict dictionary, then my main hobby is playing video games. I am a dedicated gamer. I spend hours every single evening leveling up my characters, finding rare items, and completing dangerous missions.
Nu: Oh, here we go. Listeners, we need to have our first major debate right now. Video games are not a real hobby. They are just a digital distraction. They melt your brain. You just sit in a dark room, staring at a glowing screen for hours, doing nothing in the real world. It is completely unproductive and isolates you from society.
Nam: Melt my brain? Emma, you have absolutely no idea what you are talking about. Video games do not melt my brain. They expand my brain! I am literally saving the universe on a daily basis! I am solving complex puzzles under pressure, managing virtual economies, and leading entire armies of players to victory. It requires intense strategy, quick reflexes, and teamwork. What do you do for a hobby? You probably sit in silence and organize your socks by color.
Nu: I do not organize my socks for fun. That is a chore. My hobbies are actually productive and beneficial to my health. I read non-fiction books, I practice calligraphy, and I do yoga. Reading makes you smarter and expands your worldview. Yoga makes you flexible, strong, and mentally calm. What do video games give you? Sore thumbs, a bad posture, and a high electricity bill.
Nam: Video games give me happiness! And they teach me things. Listeners, I actually learned a huge amount of English vocabulary from playing video games when I was a teenager. When you are trying to survive in a game, you learn the words for weapons, directions, and strategies very quickly. Plus, my high score is a work of art. You just don't understand the digital world, Emma.
Nu: I understand that you need to get off the sofa and see the sun. This brings me to a great phrasal verb for our listeners. "To take up" a hobby. It means to start doing a new activity. For example, David, you really need to take up a new hobby. Something that involves moving your physical body in the real world.
Nam: I move my body. I walk to the kitchen to get potato chips while my game is loading. That is movement. I also stretch my arms when I win.
Nu: I mean real exercise. This leads us to another category of hobbies: outdoor hobbies. I consider myself a very outdoorsy person. "Outdoorsy" is a great adjective used to describe someone who loves spending time outside in nature, rather than staying inside.
Nam: Oh no. I know exactly where this is going. You are going to talk about dirt and bugs.
Nu: I am going to talk about the beauty of nature. On the weekends, my absolute favorite hobby is hiking. I love packing a backpack with healthy snacks, driving up to the mountains, and hiking for five or six hours. There is nothing better than climbing a steep, difficult trail, feeling your heart pump, and finally breathing the fresh, clean air at the top of a mountain. The view is always worth the effort.
Nam: I can think of a million things better than that. Listeners, prepare for another debate. Hiking is the worst hobby in the history of the world. Why would anyone volunteer to walk up a giant hill for no reason?
Nu: It is not for no reason! It is for the spectacular view. It is for the physical challenge. It makes you feel alive. Nature is beautiful, peaceful, and majestic.
Nam: Nature is terrible, Emma. It is full of mosquitoes, mud, and dangerous wild animals. You go outside, you sweat profusely, you get sunburned, your legs hurt, and you get bitten by bugs. And for what? To look at a tree? To look at a rock? I have pictures of trees and rocks on my computer screen. They are in 4K high definition, and I do not have to get sweaty or eaten by a bear to see them.
Nu: You are being ridiculous. Getting fresh air is essential for your mental health. Being in nature reduces stress and lowers your blood pressure. When I am walking in the forest, surrounded by tall trees, I feel completely relaxed and disconnected from the stress of work.
Nam: When I am in the forest, I feel like a bear's dinner. A bear is not peaceful, Emma. You know what is peaceful? My living room. My sofa does not have bears. My sofa has air conditioning and a strong Wi-Fi connection. Those are the two greatest inventions in human history. I prefer indoor hobbies where the temperature is strictly controlled, there is no mud, and the snacks are very close to me. Hiking is simply not my cup of tea.
Nu: That is a perfect idiom, David. Listeners, when you say something is "not my cup of tea," it means you do not like it or you are not interested in it. Hiking is not David's cup of tea. He prefers to stay in his comfort zone. But you are missing out on the beauty of the real world because you are afraid of a little dirt. Fine, if you insist on indoor hobbies, what else do you do besides gaming? Do you play any musical instruments?
Nam: Instruments? Well, I tried to take up the guitar once. I bought a very nice, shiny, expensive acoustic guitar. I was very excited. I watched three YouTube tutorials, tried to play a song, my fingers started to hurt terribly, and I never touched it again. Now it is a very expensive decoration sitting in the corner of my bedroom.
Nu: That is exactly your problem, David. You give up way too easily. Learning a new hobby takes patience and consistency. You cannot expect to be an expert on the very first day. When I started learning calligraphy, which is the art of beautiful, decorative handwriting, my letters looked like a confused spider walked across the paper. It was terrible. But I practiced for twenty minutes every single day, and eventually, I got the hang of it.
Nam: "Got the hang of it." That is a good phrase. To get the hang of something means to learn how to do something, especially after practicing it. But see, that sounds exactly like a job. Practice, patience, daily effort. Hobbies are supposed to be a way to chill out. "To chill out" means to relax completely. If my hobby causes me stress and finger pain, it is not a good hobby.
Nu: A good hobby challenges you. It gives you a sense of achievement and pride. When you finally master a difficult song on the guitar, the feeling of success is incredible. You missed out on that feeling completely because you quit the moment it got hard.
Nam: I didn't quit. I just permanently postponed my musical career. Anyway, let's talk about collecting things. People love collecting things as a hobby. Stamps, coins, vintage clothes. Do you collect anything, Emma?
Nu: I do, actually. I collect vintage postcards from different countries. Whenever my friends travel, they send me one. It is a wonderful way to learn about history, geography, and different cultures. It is a very sophisticated hobby. Do you collect anything, David?
Nam: I collect digital skins for my video game characters. I have a rare glowing sword that cost me fifty dollars.
Nu: You spent real money on a fake sword? That is not a hobby, David. That is a waste of money. A collection should have historical or sentimental value. A glowing digital sword does not exist.
Nam: It exists on the server! And it makes my character look awesome. It has huge sentimental value to me. It strikes fear into the hearts of my enemies.
Nu: Your enemies are twelve-year-old kids on the internet. Let's move on to a hobby that everyone actually loves. Food.
Nam: Now we are talking. My favorite hobby is eating. I am a massive foodie. I explore new restaurants, order different types of pizza, and find the perfect chocolate cake. That is a lifestyle. It requires research, dedication, and a big stomach.
Nu: Eating is not a hobby! We literally just discussed this with sleeping. However, cooking is a fantastic hobby. I love finding new, healthy, nutritious recipes. On Sunday afternoons, I spend hours in the kitchen meal prepping. I chop fresh vegetables, I bake chicken breast, and I organize my healthy meals in little boxes for the entire week. It is a very satisfying and productive process.
Nam: Cooking is okay, but I prefer the other side of the process. Why would I spend two hours chopping onions and crying, when a nice man on a motorbike will bring a hot pizza directly to my door in thirty minutes? A hobby should not make you cry.
Nu: Onions make you cry, not the hobby! Being a foodie is fun, but a hobby should involve creating something. When I cook, I create a nutritious meal that fuels my body. When you order a pizza, you just create a dirty cardboard box and a higher cholesterol level.
Nam: I create joy in my stomach! I am supporting the local economy. But speaking of creating things, do you have any creative hobbies involving nature? Like gardening?
Nu: I actually enjoy gardening very much. I have a small balcony in my apartment, and I grow cherry tomatoes, fresh basil, and a lot of beautiful, colorful flowers. It is incredibly rewarding to plant a tiny seed in the dirt, water it, and watch it grow into something that produces food or beauty.
Nam: Gardening sounds dangerous. You are literally inviting nature into your house. Before you know it, there will be spiders and bugs on your balcony.
Nu: Bugs are a natural part of the ecosystem, David. They are not going to hurt you. Have you ever tried to keep a plant alive?
Nam: Yes, actually. I had a small cactus on my desk at work. The lady at the shop told me it was impossible to kill. She said taking care of a cactus is a piece of cake. "A piece of cake" is a fantastic idiom, listeners. It means something is extremely easy to do.
Nu: So, what happened to the poor cactus? If it was a piece of cake, it should still be alive and well.
Nam: Well, I forgot to water it for about eight months. It turned completely brown, shrunk, and became very crunchy. I guess I am not very good at gardening. I have a "black thumb" instead of a "green thumb."
Nu: A "green thumb" means you are naturally talented at growing plants. You definitely have a black thumb. You shouldn't be allowed anywhere near living plants.
Nam: I completely agree. That is exactly why I stick to digital plants in my video games. They never die, they do not need water, and they never attract insects. It is a flawless system.
Nu: You know, hobbies are actually very important for your professional career too. Employers often ask about your hobbies in job interviews. They want to see what kind of person you are outside of the office.
Nam: Really? So if they ask me in a serious job interview, "David, what are your hobbies?" what should I say? Should I tell them about my glowing digital sword?
Nu: Absolutely not. I do not think a serious company wants to hear about your digital wizard character. You should tell them about hobbies that show positive character traits, like teamwork, leadership, discipline, or creativity. For example, if you say you play a team sport like basketball, it shows you can work well with others.
Nam: Okay. What if I tell them I am the leader of my guild in my online video game? I command fifty players from around the world in complex digital battles. I organize schedules, I resolve conflicts, and I lead them to victory. That shows immense leadership and management skills, right?
Nu: While that is strangely impressive, most managers won't understand it. You should mention something more grounded in reality. Like reading non-fiction, volunteering in the community, or learning a new language.
Nam: Fine. I will tell them my hobby is learning English with Emma on the Just English Channel. That sounds very productive and intellectual.
Nu: That actually is extremely productive! Listening to podcasts to improve your language skills is a fantastic hobby. It is educational, it challenges your brain, and it will definitely help you in the real world. Listeners, we are very proud of you for making English learning one of your daily hobbies.
Nam: Yes, but a quick warning to our listeners: do not forget to have fun, relaxing hobbies too. If you only do productive, difficult things all the time, you will burn out. "To burn out" means to become completely exhausted and stressed from working too hard. You need balance. You need some time to be a couch potato.
Nu: A "couch potato" is someone who sits on the sofa all day watching television and eating junk food. It is a negative term, David. We do not want our listeners to be couch potatoes. We want them to be active.
Nam: Being a couch potato is my ultimate weekend goal. It is an art form. You need a comfortable blanket, the remote control in the exact perfect position, and an endless supply of snacks.
Nu: You need to go outside and see the sun! Look, a good, healthy life has a balance of different hobbies. You should have one hobby to keep you creative, like drawing, writing, or playing music. You should have one hobby to keep you in shape, like hiking, swimming, or going to the gym. And you should have one hobby to improve your career or your mind, like coding, reading, or learning English.
Nam: That sounds like three extra full-time jobs. I only have one life. I want to spend it enjoying myself, not constantly trying to improve myself until I collapse.
Nu: Improving yourself is enjoyable. When you see progress, when you can speak a new language or climb a higher mountain, it makes you feel proud and confident.
Nam: Winning a difficult video game makes me feel proud. Finding a new, delicious pizza place makes me feel confident. We just have completely different definitions of success and happiness, Emma.
Nu: I suppose we do. But that is exactly why we have this podcast. To show different perspectives, debate funny topics, and learn how to talk about them naturally in English. Let's do a quick summary of the important vocabulary we covered today.
Nam: Good idea. We talked about "leisure time," which is your free time to do whatever you want. Hopefully, that means taking naps.
Nu: We talked about "taking up" a hobby, which means to start a new activity. Like David taking up the guitar and quitting immediately.
Nam: Hey, I didn't quit. I am on an extended break. Then we had the idiom "a piece of cake," meaning something is very, very easy. Like killing a cactus.
Nu: We discussed being "outdoorsy," which means loving nature and outdoor activities like hiking. And we talked about being a "couch potato," which is David's lifestyle of sitting inside and doing absolutely nothing.
Nam: It is a peaceful, bug-free lifestyle. We also learned the phrase "not my cup of tea," which means you don't like something. Hiking is not my cup of tea. We also learned "to chill out," meaning to relax, and "to burn out," which means to get too tired from working too hard.
Nu: Exactly. Now it is time for our viewer challenge. Listeners, we want to hear from you in the comments section below. What are your favorite hobbies?
Nam: Are you Team Emma, running up dangerous mountains, getting bitten by bugs, and eating green leaves? Or are you Team David, chilling comfortably on the sofa, eating pizza, and saving the digital universe? Let us know. We want to see who has the best hobbies.
Nu: Please do not call a healthy salad "leaves." Tell us what you actually like to do in your leisure time. Try to use some of the vocabulary we learned today. For example, tell us if you recently took up a new hobby, or if there is a hobby that is simply not your cup of tea.
Nam: And if your hobby is eating pizza and taking naps, you are officially my best friend.
Nu: Don't forget to like this video, subscribe to the Just English Channel, and hit the notification bell so you never miss an episode. We truly love reading your comments and interacting with you all.
Nam: Yes, subscribing to our channel is the easiest and best hobby you can take up today. It takes one second. It is a piece of cake. Thank you all so much for listening.
Nu: Have a wonderful, productive week, everyone. Stay active and find time for the things you love.
Nam: Goodbye everyone! I am going to practice my favorite hobby now. Goodnight!
Nu: David, it is two o'clock in the afternoon! You cannot go to sleep!
Nam: Time is just an illusion, Emma! Bye!
"""
# 👆👆👆 ========================================================== 👆👆👆

file_nam = "nam_ref.wav"
file_nu = "nu_ref.wav"

if not os.path.exists(file_nam) or not os.path.exists(file_nu):
    print("⚠️ LỖI: Không tìm thấy file giọng mẫu. Hãy chạy Bước 2 trước!")
else:
    model = load_model("CLONE")
    final_audio = AudioSegment.silent(duration=500)
    gap = AudioSegment.silent(duration=400)

    lines = [l for l in kich_ban.strip().split('\n') if ":" in l]
    total_lines = len(lines)

    print("⚡️ Đang học giọng Nam & Nữ (Chỉ làm 1 lần)...")
    nam_prompt_feature = model.create_voice_clone_prompt(ref_audio=file_nam, ref_text=None, x_vector_only_mode=True)
    nu_prompt_feature = model.create_voice_clone_prompt(ref_audio=file_nu, ref_text=None, x_vector_only_mode=True)

    print("🚀 BẮT ĐẦU THU ÂM PODCAST...")
    for index, line in enumerate(lines):
        name_part, text = line.split(":", 1)
        clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower()
        text = text.strip()

        # Nhận diện Nam/Nữ (Đã cập nhật đủ chữ david, emma)
        is_nam = clean_name in ["nam", "man", "male", "host", "teacher", "eric", "ryan", "mr", "david"]
        current_prompt = nam_prompt_feature if is_nam else nu_prompt_feature

        if text:
            print(f"🎙️ [{index+1}/{total_lines}] {clean_name.upper()}: Đang thu âm...")
            with torch.inference_mode():
                w, sr = model.generate_voice_clone(text=text, voice_clone_prompt=current_prompt)

            sf.write("temp.wav", w[0], sr)
            final_audio += AudioSegment.from_wav("temp.wav") + gap
            if os.path.exists("temp.wav"): os.remove("temp.wav")

    # Xuất file
    final_audio.export("Podcast_ThanhPham.mp3", format="mp3")
    clear_output()
    print("🎉 HOÀN TẤT! Đã ghép nối xong toàn bộ kịch bản.")
    print("👇 Bấm Play để nghe hoặc bấm dấu 3 chấm (⋮) để Tải xuống MP3 👇")
    display(Audio("Podcast_ThanhPham.mp3"))

🎉 HOÀN TẤT! Đã ghép nối xong toàn bộ kịch bản.
👇 Bấm Play để nghe hoặc bấm dấu 3 chấm (⋮) để Tải xuống MP3 👇
